# SSL vs. ImageNet vs. Scratch — significance testing

Design recap (as I understand it):
- 5 classes total. For every split, 3 classes are used for pre-training (SSL) and the remaining 2 for fine-tuning → C(5,3) = **10 splits**, matching the 10 rows per method in the CSV.
- 4 initializations: `scratch`, `ImageNet`, `SSL-SimCLR`, `SSL-BYOL`.
- Each split × method combination was evaluated at 4 fine-tuning data fractions: 5%, 10%, 15%, 20%.

Two questions:
1. **Does the average of the two SSL methods beat ImageNet init?** → paired permutation test.
2. **Are some splits systematically harder/easier than others?** → Friedman test.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, permutation_test

RNG = np.random.default_rng(0)

df = pd.read_csv('/mnt/user-data/uploads/results_per_split.csv')
df.columns = ['split_label', 'method', 5, 10, 15, 20]

# split index (0-9) pulled out of the label, method name cleaned up
df['split'] = df['split_label'].str.extract(r'^(\d+):').astype(int)
df['method'] = df['method'].str.replace(r'^\d+:', '', regex=True)

long = df.melt(id_vars=['split', 'method'], value_vars=[5, 10, 15, 20],
               var_name='fraction', value_name='score')
long = long.sort_values(['method', 'split', 'fraction']).reset_index(drop=True)
long.head()


: 

## Configuration: which fine-tuning fraction(s) to analyze

Set `FRACTION_MODE` below to control what a "score" means for the rest of the
notebook:
- `'average'` — average each split/method's score over all 4 fractions (5, 10,
  15, 20%) before running any test. This is what both analyses below used by
  default previously.
- `5`, `10`, `15`, or `20` — use only that single fine-tuning-fraction column,
  ignoring the others entirely.

Everything in sections 1 and 2 re-runs automatically off of whichever mode you
pick here. The appendix cells at the end of each section still show the
full, all-fractions breakdown regardless of this setting, in case you want the
fuller picture alongside the single number.


In [ ]:
# --- CONFIG: pick one ---
FRACTION_MODE = 5 # 'average'   # 'average', or one of 5, 10, 15, 20

def get_scores(data_long, mode):
    """Return one score per (split, method): either the single chosen
    fraction column, or the average across all 4 fraction columns."""
    if mode == 'average':
        out = data_long.groupby(['split', 'method'])['score'].mean().reset_index()
    else:
        if mode not in data_long['fraction'].unique():
            raise ValueError(f"mode must be 'average' or one of {sorted(data_long['fraction'].unique())}")
        out = data_long[data_long['fraction'] == mode][['split', 'method', 'score']].reset_index(drop=True)
    return out

scores = get_scores(long, FRACTION_MODE)
label = 'averaged over all fractions' if FRACTION_MODE == 'average' else f'{FRACTION_MODE}% fine-tuning data only'
print(f"Using: {label}  ({len(scores)} split/method rows)")
scores.head()


## 1. Permutation test: ImageNet vs. mean(SSL-SimCLR, SSL-BYOL)

**Is a permutation test a reasonable choice here?** Yes — with only 10 splits, a
parametric test (paired t-test) would lean heavily on a normality assumption you
can't really check with n=10, and several scores are pinned near 1.0 (ceiling
effects), which makes normality even less plausible. A permutation test makes no
distributional assumption and is exact (or near-exact) at this sample size, so
it's a good fit.

**One thing to decide carefully: what counts as "one independent unit"?**
Your two ImageNet vs. SSL scores are *paired* at the (split, fraction) level —
they were computed from the same fine-tuning subset. But the 4 fraction levels
within a split are *not* independent of each other (they come from nested/overlapping
subsets of the same fine-tuning data for that split). Using `FRACTION_MODE` above
sidesteps this cleanly either way:

- `FRACTION_MODE = 'average'` → one score per split per method (n=10 paired
  differences), which is the safer choice for validity since the split — the
  actual cross-validation fold — becomes the independent unit.
- `FRACTION_MODE = 5/10/15/20` → also n=10, but tells you whether SSL's edge
  holds up at that specific data amount rather than "on average".

The appendix at the end of this section also shows the pooled (n=40, all
fraction-pairs at once) and per-fraction breakdown together, if you want to see
how sensitive the result is to that choice.


In [ ]:
def paired_sign_flip_test(x, y, n_resamples=100_000, alternative='two-sided', random_state=RNG):
    """Paired permutation (sign-flip) test on x - y.
    Exact enumeration is used automatically by scipy when n is small enough
    for the chosen alternative; otherwise falls back to Monte Carlo.
    """
    diff = np.asarray(x) - np.asarray(y)

    def statistic(d, axis=-1):
        return np.mean(d, axis=axis)

    res = permutation_test(
        (diff,), statistic, permutation_type='samples',
        n_resamples=n_resamples, alternative=alternative, random_state=random_state
    )
    return res.statistic, res.pvalue

# --- prep, using the score defined by FRACTION_MODE above ---
ssl_methods = ['SSL-SimCLR', 'SSL-BYOL']
ssl_avg = (scores[scores['method'].isin(ssl_methods)]
           .groupby('split')['score'].mean().reset_index()
           .rename(columns={'score': 'ssl_avg'}))
imagenet = scores[scores['method'] == 'ImageNet'][['split', 'score']] \
    .rename(columns={'score': 'imagenet'})

per_split = ssl_avg.merge(imagenet, on='split')
per_split.head()


In [ ]:
# --- Main test, driven by FRACTION_MODE (n=10 paired differences) ---
stat, p = paired_sign_flip_test(per_split['ssl_avg'], per_split['imagenet'])
print(f"[{label}] mean diff (SSL-avg - ImageNet) = {stat:.4f}, p = {p:.4f}")


### Appendix: full fraction-level breakdown (ignores `FRACTION_MODE`)

This always uses every (split, fraction) pair regardless of the config above,
so you can see the pooled result and the per-fraction pattern side by side with
whatever single number the main test above gave you.


In [ ]:
ssl_long = long[long['method'].isin(ssl_methods)]
ssl_avg_all = (ssl_long.groupby(['split', 'fraction'])['score']
               .mean().reset_index().rename(columns={'score': 'ssl_avg'}))
imagenet_all = long[long['method'] == 'ImageNet'][['split', 'fraction', 'score']] \
    .rename(columns={'score': 'imagenet'})
paired_all = ssl_avg_all.merge(imagenet_all, on=['split', 'fraction'])

# Pooled (n=40 pairs, all fractions at once)
stat_pooled, p_pooled = paired_sign_flip_test(paired_all['ssl_avg'], paired_all['imagenet'])
print(f"Pooled (n=40 pairs): mean diff = {stat_pooled:.4f}, p = {p_pooled:.4f}")

# Per-fraction (n=10 each)
print("\nPer-fraction:")
for frac, g in paired_all.groupby('fraction'):
    s, p_f = paired_sign_flip_test(g['ssl_avg'], g['imagenet'])
    print(f"  {frac:>2}% fine-tuning data: mean diff = {s:.4f}, p = {p_f:.4f}")


> **Note on interpretation:** I used a two-sided alternative above. If your
> hypothesis is directional — you expect domain SSL to *beat* ImageNet, not just
> "differ from" it — a one-sided test (`alternative='greater'`) is more
> appropriate and roughly halves the p-value for a given effect direction. I
> didn't want to assume that choice for you.


## 2. Friedman test: are some splits harder/easier than others?

**Is Friedman a reasonable choice?** Yes, but the setup needs to be right. Friedman
tests whether a set of "treatments" are ranked consistently across "blocks". Here
you want the **splits to be the treatments** (10 of them) and the **methods to be
the blocks** — each method gives one ranking of the 10 splits by score, and
Friedman checks whether these rankings agree more than chance would allow. A
significant result means some splits are consistently easier/harder across
methods — which is exactly what you asked. This also uses `FRACTION_MODE` from
the config above, so each method's ranking is based on either the
average-over-fractions score or the single chosen fraction, consistent with
section 1.

Caveat: this treats `scratch` runs as an informative block too. If you only
care about difficulty *for the methods you're actually comparing*, drop
`scratch` and use 3 blocks (ImageNet, SimCLR, BYOL) instead — I show both below.
With only 3–4 blocks this test has limited power, so a non-significant result
here shouldn't be read as "splits are equally hard" — just that 3–4 rankings
isn't a lot of evidence either way. The appendix uses all 4 fractions as
separate blocks instead (more blocks, more power), at the cost of those blocks
not being fully independent of one another — see the section 1 discussion.


In [ ]:
def friedman_on_splits(scores_df, methods):
    sub = scores_df[scores_df['method'].isin(methods)].copy()
    # one row per block (method), one column per split
    wide = sub.pivot_table(index='method', columns='split', values='score')
    wide = wide.sort_index(axis=1)  # splits 0..9 in order
    stat, p = friedmanchisquare(*[wide[c].values for c in wide.columns])
    return stat, p, wide

# All 4 methods as blocks
stat_all, p_all, wide_all = friedman_on_splits(scores, scores['method'].unique())
print(f"[{label}] Friedman (incl. scratch, {wide_all.shape[0]} blocks): "
      f"chi2 = {stat_all:.3f}, p = {p_all:.4f}")

# Only the 3 'real' comparison methods - drop scratch
stat_ns, p_ns, wide_ns = friedman_on_splits(scores, ['ImageNet', 'SSL-SimCLR', 'SSL-BYOL'])
print(f"[{label}] Friedman (excl. scratch, {wide_ns.shape[0]} blocks): "
      f"chi2 = {stat_ns:.3f}, p = {p_ns:.4f}")


### Appendix: all 4 fractions as separate Friedman blocks (ignores `FRACTION_MODE`)

This is the higher-power version that uses every (method × fraction) combination
as its own block (4 methods × 4 fractions = 16 blocks), regardless of the config
setting above.


In [ ]:
def friedman_on_splits_all_fractions(data_long, methods):
    sub = data_long[data_long['method'].isin(methods)].copy()
    wide = sub.pivot_table(index=['method', 'fraction'], columns='split', values='score')
    wide = wide.sort_index(axis=1)
    stat, p = friedmanchisquare(*[wide[c].values for c in wide.columns])
    return stat, p, wide

stat_16, p_16, wide_16 = friedman_on_splits_all_fractions(long, long['method'].unique())
print(f"Friedman (incl. scratch, {wide_16.shape[0]} blocks): chi2 = {stat_16:.3f}, p = {p_16:.4f}")

stat_12, p_12, wide_12 = friedman_on_splits_all_fractions(
    long, ['ImageNet', 'SSL-SimCLR', 'SSL-BYOL'])
print(f"Friedman (excl. scratch, {wide_12.shape[0]} blocks): chi2 = {stat_12:.3f}, p = {p_12:.4f}")


In [ ]:
# Rank splits by mean rank across the 16 (method x fraction) blocks
# (higher rank = easier / higher score)
ranks = wide_16.rank(axis=1)
mean_rank = ranks.mean(axis=0).sort_values()
mean_score = wide_16.mean(axis=0).loc[mean_rank.index]

summary = pd.DataFrame({'mean_rank': mean_rank, 'mean_score': mean_score})
print(summary)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(summary.index.astype(str), summary['mean_score'], color='#4C72B0')
ax.set_xlabel('Split')
ax.set_ylabel('Mean score (across methods & fractions)')
ax.set_title('Average difficulty per split (lower bar = harder split)')
plt.tight_layout()
plt.show()


## Optional: pairwise post-hoc test between splits

If the Friedman test is significant and you want to know *which* specific splits
differ from which others (not just "some split differs somewhere"), the standard
follow-up is a **Nemenyi post-hoc test** on the same rank data. It's not in scipy;
the `scikit-posthocs` package implements it (`pip install scikit-posthocs`).
I haven't run this myself in the container since the package isn't installed
here, so treat the snippet below as untested — please double check the output
before relying on it.


In [ ]:
# pip install scikit-posthocs
# import scikit_posthocs as sp
# nemenyi = sp.posthoc_nemenyi_friedman(wide_16.values)
# nemenyi.index = nemenyi.columns = wide_16.columns
# nemenyi  # matrix of pairwise p-values between splits
